# Пробный прогон T-lite: эпизод целиком

Исходная T-lite-it-2.1 (без обучения) с системным промптом `prompts.py`: промпт → генерация → вызовы Википедии →
ответ → разбор DSL → PPTX. Смотрим, держит ли модель формат без примера в промпте, ищет ли сама, сколько занимает
эпизод по токенам, времени и памяти. Запуск — на сервере (V100 32 ГБ, float16).

Результаты — `runs/tlite_pilot/<время>/`: для каждого задания весь эпизод текстом (`<id>.txt`), ответ для парсера
(`<id>_answer.txt`), презентация (`<id>.pptx`) и сводка `summary.json`.

In [ ]:
import dataclasses, json, os, sys, time
from pathlib import Path

# Сервер общий: показываем модели одну свободную карту (свободные — по nvidia-smi), иначе device_map="auto"
# займёт память на всех картах. Строка должна стоять до import torch.
os.environ["CUDA_VISIBLE_DEVICES"] = "4"

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))  # если пакет не установлен через pip install -e .
import torch
from presentation_rl.dsl import parse
from presentation_rl.environment import build_prompt, run_episode
from presentation_rl.model_loader import hf_generator, load_model
from presentation_rl.prompts import system_prompt
from presentation_rl.rendering import build_presentation, pick_theme
from presentation_rl.tasks import load_tasks
from presentation_rl.tools.cache import WikiCache
from presentation_rl.tools.wiki import WikiClient
from presentation_rl.tools.wiki_tools import WikiTools

print(torch.__version__, torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))
CONTACT = os.environ.get("WIKI_CONTACT", "")  # или вписать сюда почту / ссылку на проект
cache = WikiCache(ROOT / "data" / "sources" / "wiki_cache", WikiClient(contact=CONTACT))
OUT = ROOT / "runs" / "tlite_pilot" / time.strftime("%Y%m%d_%H%M")
OUT.mkdir(parents=True, exist_ok=True)

In [ ]:
start = time.time()
model, tokenizer = load_model("t-tech/T-lite-it-2.1", dtype="float16")
generate = hf_generator(model, tokenizer, temperature=0.9)
print(f"загрузка {time.time() - start:.0f} с, память {torch.cuda.memory_allocated() / 2**30:.1f} ГБ")

In [ ]:
# Четыре задания validation и четыре сочетания полей «Поиск» и «Картинки».
tasks = {t.id: t for t in load_tasks(ROOT / "data" / "tasks" / "tasks.json", split="validation")}
RUNS = [("t367", True, True), ("t366", True, False), ("t369", False, True), ("t370", False, False)]

In [ ]:
summary = []
for task_id, search, images in RUNS:
    request = dataclasses.replace(tasks[task_id].request, search=search, images=images)
    tools = WikiTools(cache) if search else None
    prompt_ids = build_prompt(tokenizer, system_prompt(request), request)
    torch.cuda.reset_peak_memory_stats()
    start = time.time()
    episode = run_episode(prompt_ids, tokenizer, generate, tools, max_new_tokens=8000)
    seconds = time.time() - start
    deck = parse(episode.answer)
    report = build_presentation(deck, OUT / f"{task_id}.pptx", pick_theme(request.theme, request.text),
                                sources=tools.source if tools else None, show_sources=request.sources)
    (OUT / f"{task_id}.txt").write_text(tokenizer.decode(episode.ids), encoding="utf-8")
    (OUT / f"{task_id}_answer.txt").write_text(episode.answer, encoding="utf-8")
    row = {
        "id": task_id, "search": search, "images": images, "finish": episode.finish,
        "prompt_tokens": len(episode.prompt_ids), "generated_tokens": episode.generated_tokens,
        "total_tokens": len(episode.ids), "calls": [f"{c['name']}({c['argument']})" for c in episode.calls],
        "parse_errors": len(deck.errors), "error_codes": sorted({e.code for e in deck.errors}),
        "slides": report.slide_count, "render": report.status_counts(), "seconds": round(seconds),
        "peak_memory_gb": round(torch.cuda.max_memory_allocated() / 2**30, 1),
    }
    summary.append(row)
    print(json.dumps(row, ensure_ascii=False))
    # сводка — после каждого задания, чтобы при обрыве не пропала
    (OUT / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=1), encoding="utf-8")

In [ ]:
# Первый эпизод с поиском целиком — как его видела модель (без системного промпта).
text = (OUT / "t367.txt").read_text(encoding="utf-8")
print(text[text.index("<|im_start|>user"):])